# מפגש 17 — פתרון מורה Audited
מחברת התלמיד המקורית: bill_length_mm/bill_depth_mm, seed17. אינה תחליף להרצת Run All ב־Colab. היא מכילה בדיקות לוולידציה, אינדקס ו־Accuracy.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

url = "https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
df = pd.read_csv(url)
print("Source:", df.shape)
display(df.head(3))
display(df.isna().sum())
assert df.shape == (344,7)
assert df['species'].notna().all()

## 1. X/y ואימון/בדיקה משותפים
תכונות מקור במ״מ; species אינה מאפיין. פיצול 25% עם stratify ו־seed17.

In [ ]:
model_df = df.dropna(subset=["species"]).copy()
features = ["bill_length_mm","bill_depth_mm"]
X = model_df[features].copy()
y = model_df["species"].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print("Train:", X_train.shape, "Test:", X_test.shape)
assert df.shape == (344, 7)
assert X_train.shape == (258, 2)
assert X_test.shape == (86, 2)
assert X_train.index.equals(y_train.index)
assert X_test.index.equals(y_test.index)
assert "species" not in X.columns

## 2. מילוי ממוצע ו־StandardScaler — Train בלבד
אין לחלק פרמטרים בין קבוצות ואין להפעיל scaler.fit על Test.

In [ ]:
training_means = X_train.mean()
X_train_ready = X_train.fillna(training_means)
X_test_ready = X_test.fillna(training_means)
scaler = StandardScaler().set_output(transform="pandas")
scaler.fit(X_train_ready)
X_train_scaled = scaler.transform(X_train_ready)
X_test_scaled = scaler.transform(X_test_ready)
display(X_train_scaled.head(3))
assert X_train_scaled.shape == X_train.shape
assert X_test_scaled.shape == X_test.shape
assert X_train_scaled.index.equals(X_train.index)
assert X_test_scaled.index.equals(X_test.index)
assert not X_train_scaled.isna().any().any()
assert not X_test_scaled.isna().any().any()
assert np.allclose(scaler.mean_, X_train_ready.mean().to_numpy())
assert X_train_scaled.columns.tolist() == features

## 3. KNN K=3 — fit/predict
קביעת K מראש; לצורכי הוראה אין תהליך בחירת פרמטרים לפי תוצאות Test.

In [ ]:
model = KNeighborsClassifier(n_neighbors=3, weights="uniform", metric="euclidean")
model.fit(X_train_scaled, y_train)
predictions = model.predict(X_test_scaled)
print("First five predictions:", predictions[:5])
assert len(predictions) == len(y_test)
assert set(predictions).issubset(set(y_train.unique()))
assert model.n_neighbors == 3
assert model.weights == 'uniform'
assert model.metric == 'euclidean'

## 4. Accuracy — ידני מול ספרייה
מחשבים על אותו סדר שורות ואותו אינדקס של y_test.

In [ ]:
comparison = pd.DataFrame(
    {"actual": y_test, "predicted": predictions},
    index=y_test.index
)
comparison["correct"] = comparison["actual"] == comparison["predicted"]
correct_count = int(comparison["correct"].sum())
accuracy_hand = correct_count / len(comparison)
accuracy_library = accuracy_score(y_test, predictions)
print("Correct:", correct_count, "/", len(comparison),
      "Accuracy:", accuracy_library, "Percent:", 100 * accuracy_library)
display(comparison.head(10))
assert comparison.index.equals(y_test.index)
assert len(comparison) == 86
assert abs(accuracy_hand - accuracy_library) < 1e-12
assert 0 <= accuracy_library <= 1
assert correct_count + int((~comparison['correct']).sum()) == len(y_test)

## 5. טעויות בפועל ומדידות שמקושרות לפי אינדקס
מציגים עד שלוש. אם אין טעויות נציג ריק ולא נייצר שורות מומצאות.

In [ ]:
errors = comparison[comparison["correct"] == False].copy()
print("Errors observed:", len(errors))
display(errors.head(3))
if len(errors) > 0:
    display(X_test.loc[errors.head(3).index])
else:
    print("No errors to display in this split.")
assert correct_count + len(errors) == len(comparison)
assert all(errors["actual"] != errors["predicted"])
assert errors.index.isin(X_test.index).all()

## 6. בדיקת תהליך, תוויות ושמירת מקור
ה־accuracy הוא דיוק על חלוקה אחת, לא הבטחה לזמן עתידי או לאוכלוסייה אחרת.

In [ ]:
assert features == ["bill_length_mm","bill_depth_mm"]
assert model_df.shape[1] == 7
assert "species" not in X.columns
assert len(X_train) + len(X_test) == len(model_df)
assert X_test.index.equals(y_test.index)
assert X_train.index.equals(y_train.index)
assert correct_count + len(errors) == 86
assert abs(accuracy_hand - accuracy_library) < 1e-12
print("Model card: bill measurements, K=3, Train-only scaling, one held-out test split.")

## סיכום למורה
במצגת ההכנה אין לחשוף את תוצאות המטלה. שאלו מדוע אין להשתמש ב־y_test בזמן חיזוי, מהי מגבלת Accuracy והאם הבדל בין Seeds מאפשר להשוות זוג מאפיינים. Run All בענן והגהת מצגת עוד לא בוצעו.